# Prétraitement Avancé et Modélisation

## Chargement des données

In [24]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("gestionlogistique").getOrCreate()
df = spark.read.option("header", "true").csv("data/output_tmp/part-00000-039c67c4-4607-444e-be84-ab46ccdf438b-c000.csv")

df.show()
df.printSchema()

+-------------+------------------------+-----------------------------+-----------------+------------------+------------------+---------------+-------------------+------------------------+-------------------+----------------+----------------------+--------+----------------+----------------+--------------+--------------------+---------------+--------------+------------------+
|Order Country|Days for shipping (real)|Days for shipment (scheduled)|Benefit per order|Sales per customer|Late_delivery_risk|Department Name|Order Item Quantity|Order Item Product Price|Order Item Discount|Order Item Total|Order Profit Per Order|    Type| Delivery Status|   Category Name|Customer State|         Order State|   Order Region| Shipping Mode|          distance|
+-------------+------------------------+-----------------------------+-----------------+------------------+------------------+---------------+-------------------+------------------------+-------------------+----------------+----------------------

## Conversion des colonnes numériques

In [4]:
from pyspark.sql import functions as f

numbers_columns = [
    'Benefit per order',
    'Sales per customer',
    'Order Item Quantity',
    'Order Item Product Price',
    'Order Item Discount',
    'Order Item Total',
    'Order Profit Per Order',
    'distance',
    'Late_delivery_risk'
]

for c in numbers_columns:
    df = df.withColumn(c, f.col(c).cast("double"))


## Vérification des valeurs nulles

In [5]:
for i in df.columns:
    null_count = df.filter(df[i].isNull()).count()
    print(f"{i}: {null_count} valeurs nulles")

Order Country: 0 valeurs nulles
Days for shipping (real): 0 valeurs nulles
Days for shipment (scheduled): 0 valeurs nulles
Benefit per order: 0 valeurs nulles
Sales per customer: 0 valeurs nulles
Late_delivery_risk: 0 valeurs nulles
Department Name: 0 valeurs nulles
Order Item Quantity: 0 valeurs nulles
Order Item Product Price: 0 valeurs nulles
Order Item Discount: 0 valeurs nulles
Order Item Total: 0 valeurs nulles
Order Profit Per Order: 0 valeurs nulles
Type: 0 valeurs nulles
Delivery Status: 0 valeurs nulles
Category Name: 0 valeurs nulles
Customer State: 0 valeurs nulles
Order State: 0 valeurs nulles
Order Region: 0 valeurs nulles
Shipping Mode: 0 valeurs nulles
distance: 0 valeurs nulles


## Analyse des variables catégorielles

In [6]:
Categorical_columns = ["Order Country","Type","Delivery Status","Customer State","Order State","Order Region","Shipping Mode","Department Name","Category Name"]
for caegorie in Categorical_columns:
    count_categorie = df.select(caegorie).distinct().count()
    print(f"{caegorie} : {count_categorie}")

Order Country : 164
Type : 4
Delivery Status : 4
Customer State : 46
Order State : 1084
Order Region : 23
Shipping Mode : 4
Department Name : 11
Category Name : 50


## Suppression des colonnes inutiles

In [7]:
df = df.drop("Order State")
df = df.drop("Days for shipping (real)")
df = df.drop("Days for shipment (scheduled)")
df = df.drop("Delivery Status")

## Mise à jour de la liste des colonnes catégorielles

In [8]:
Categorical_columns = ["Order Country","Type","Customer State","Order Region","Shipping Mode","Department Name","Category Name"]

## Encodage des variables catégorielles (One-Hot Encoding)

In [9]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

stages = []

for col_name in Categorical_columns:
    indexer = StringIndexer(inputCol=col_name, outputCol=col_name + "_index")
    stages.append(indexer)

    encoder = OneHotEncoder(inputCol=col_name + "_index", outputCol=col_name + "_ohe")
    stages.append(encoder)

pipeline = Pipeline(stages=stages)
df_encoded = pipeline.fit(df).transform(df)


## Suppression des colonnes originales et indexées

In [10]:
cols_to_drop = []
for col_name in Categorical_columns:
    cols_to_drop.append(col_name)             
    cols_to_drop.append(col_name + "_index") 

df_final = df_encoded.drop(*cols_to_drop)

## Conversion en Pandas pour analyse des outliers

In [11]:
df_numeric = df_final.select(numbers_columns).toPandas()

## Détection des valeurs aberrantes (IQR)

In [12]:
outliers_count = {}

for col in numbers_columns:
    Q1 = df_numeric[col].quantile(0.25)
    Q3 = df_numeric[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_numeric[(df_numeric[col] < lower_bound) | (df_numeric[col] > upper_bound)]
    
    outliers_count[col] = len(outliers)

for col, count in outliers_count.items():
    print(f"{col}: {count} outliers")


Benefit per order: 18932 outliers
Sales per customer: 1943 outliers
Order Item Quantity: 0 outliers
Order Item Product Price: 2048 outliers
Order Item Discount: 7533 outliers
Order Item Total: 1943 outliers
Order Profit Per Order: 18932 outliers
distance: 0 outliers
Late_delivery_risk: 0 outliers


## Transformation logarithmique des variables avec outliers

In [13]:
from pyspark.sql.functions import col, log1p , when

cols = ['Benefit per order', 'Order Profit Per Order', 'Sales per customer', 
        'Order Item Product Price', 'Order Item Discount', 'Order Item Total']

for c in cols:
    df = df_final.withColumn(c, when(col(c) < 0, 0).otherwise(col(c))) \
           .withColumn(c, log1p(col(c)))

## Normalisation des features numériques (StandardScaler)

In [14]:
from pyspark.ml.feature import StandardScaler, VectorAssembler

numeric_features = [
    "Benefit per order",
    "Sales per customer",
    "Order Item Quantity",
    "Order Item Product Price",
    "Order Item Discount",
    "Order Item Total",
    "Order Profit Per Order",
    "distance",
]

numeric_assembler = VectorAssembler(inputCols=numeric_features, outputCol="numeric_features")
df = numeric_assembler.transform(df)

scaler = StandardScaler(inputCol="numeric_features", outputCol="scaled_numeric_features", withStd=True, withMean=True)
scaler_model = scaler.fit(df)
df = scaler_model.transform(df)

print("Features numériques normalisées avec StandardScaler")

Features numériques normalisées avec StandardScaler


## Exportation du DataFrame en un fichier CSV unique pour l’entraînement

In [17]:
df.repartition(1) \
  .write \
  .mode("overwrite") \
  .parquet("data/output_training_parquet")